In [ ]:
#!uv pip install google-cloud-storage 
#!uv pip install google-auth

Audited 1 package in 63ms


In [6]:
# Run these first in a separate cell
!gcloud auth application-default login
!gcloud config set project seraphic-ripple-476917-i5

'gcloud' is not recognized as an internal or external command,
operable program or batch file.
'gcloud' is not recognized as an internal or external command,
operable program or batch file.


In [5]:


# Add this function to your script
def verify_gcs_access():
    """Verify GCS access before running the pipeline"""
    try:
        storage_client = storage.Client(project=Config.PROJECT_ID)
        bucket = storage_client.bucket(Config.SOURCE_BUCKET_NAME)
        blobs = list(bucket.list_blobs(prefix=Config.SOURCE_PREFIX, max_results=1))
        logger.info(f"GCS access verified successfully. Found {len(blobs)} files in {Config.SOURCE_PREFIX}")
        return storage_client
    except Exception as e:
        logger.error(f"Failed to verify GCS access: {e}")
        return None

# Then in your run_pipeline function
storage_client = verify_gcs_access()



NameError: name 'logger' is not defined

In [4]:
import os
import tempfile
import shutil
import subprocess
import time
import logging
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
from google.cloud import storage

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("mineru_pipeline.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger("mineru-pipeline")

# Configuration (modify these values directly when running in a notebook)
class Config:
    # GCS Configuration
    PROJECT_ID = "seraphic-ripple-476917-i5"
    SOURCE_BUCKET_NAME = "raw_papers"
    DESTINATION_BUCKET_NAME = "raw_papers"  # Same bucket, different prefix 
    SOURCE_PREFIX = "papers/"
    DEST_PREFIX = "processed/"
    
    # Local directories
    LOCAL_TEMP_DIR = Path("./temp_papers")
    LOCAL_OUTPUT_DIR = Path("./processed_papers")
    
    # Processing settings
    BATCH_SIZE = 10  # Number of papers to process in a single batch
    MAX_WORKERS = 4  # Number of download/upload workers
    MINERU_WORKERS = 4  # Number of MinerU workers for processing
    
    # Track processed files
    PROCESSED_LOG = Path("processed_files.txt")
    
    # Flag to control cleanup
    DO_CLEANUP = True

def ensure_auth():
    """Ensure GCS authentication is set up correctly"""
    try:
        # Try to create a client to test authentication
        storage.Client(project=Config.PROJECT_ID)
        logger.info("GCS authentication successful")
        return True
    except Exception as e:
        logger.error(f"GCS authentication failed: {e}")
        logger.info("Attempting to authenticate with gcloud...")
        
        # Try to authenticate using gcloud
        try:
            subprocess.run(
                "gcloud auth application-default login", 
                shell=True, 
                check=True
            )
            subprocess.run(
                f"gcloud config set project {Config.PROJECT_ID}",
                shell=True,
                check=True
            )
            logger.info("Authentication successful using gcloud")
            return True
        except subprocess.CalledProcessError:
            logger.error("Failed to authenticate using gcloud")
            return False

def get_already_processed():
    """Get set of already processed files from log"""
    already_processed = set()
    if Config.PROCESSED_LOG.exists():
        with open(Config.PROCESSED_LOG, "r") as f:
            already_processed = set(line.strip() for line in f)
        logger.info(f"Found {len(already_processed)} already processed papers")
    return already_processed

def record_processed(paper_names):
    """Record processed paper names to log file"""
    with open(Config.PROCESSED_LOG, "a") as f:
        for name in paper_names:
            f.write(f"{name}\n")
    logger.debug(f"Recorded {len(paper_names)} papers to processed log")

def download_blob(blob_info):
    """Download a single blob from GCS"""
    blob_name, client, bucket = blob_info
    try:
        blob = bucket.blob(blob_name)
        local_path = Config.LOCAL_TEMP_DIR / Path(blob_name).name
        blob.download_to_filename(local_path)
        logger.debug(f"Downloaded {blob_name} to {local_path}")
        return local_path
    except Exception as e:
        logger.error(f"Error downloading {blob_name}: {e}")
        return None

def upload_folder(upload_info):
    """Upload all files in a folder to GCS"""
    local_folder_path, paper_name, client, bucket = upload_info
    local_folder = Path(local_folder_path)
    
    # Only proceed if the folder exists and contains files
    if not local_folder.exists() or not any(local_folder.iterdir()):
        logger.warning(f"No processed files found for {paper_name}")
        return 0
    
    uploaded_count = 0
    for file_path in local_folder.glob("**/*"):
        if file_path.is_file():
            try:
                # Create relative path for destination
                rel_path = file_path.relative_to(Config.LOCAL_OUTPUT_DIR)
                dest_blob_name = f"{Config.DEST_PREFIX}{paper_name}/{rel_path}"
                
                # Upload the file
                blob = bucket.blob(dest_blob_name)
                blob.upload_from_filename(file_path)
                uploaded_count += 1
                logger.debug(f"Uploaded {file_path} to {dest_blob_name}")
            except Exception as e:
                logger.error(f"Error uploading {file_path}: {e}")
    
    return uploaded_count

def process_papers_batch(pdf_paths):
    """Process a batch of papers with MinerU using vLLM backend"""
    if not pdf_paths:
        return []
    
    # Filter out None values (failed downloads)
    valid_paths = [p for p in pdf_paths if p is not None]
    if not valid_paths:
        logger.warning("No valid PDF paths to process in batch")
        return []
    
    # Prepare input paths string for MinerU
    input_paths_str = " ".join([str(path) for path in valid_paths])
    
    # Run MinerU command with vLLM backend
    cmd = (f"mineru -p {input_paths_str} -o {Config.LOCAL_OUTPUT_DIR} -b vlm-vllm-engine "
           f"--batch --workers {Config.MINERU_WORKERS}")
    logger.info(f"Running MinerU batch: {len(valid_paths)} papers")
    logger.debug(f"MinerU command: {cmd}")
    
    start_time = time.time()
    try:
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        if result.returncode != 0:
            logger.error(f"Error processing batch: {result.stderr}")
            return []
        
        elapsed = time.time() - start_time
        logger.info(f"Batch processed in {elapsed:.2f} seconds")
        
        # Return list of base names (without extension) that were processed
        processed = [path.stem for path in valid_paths]
        logger.debug(f"Processed papers: {processed}")
        return processed
    except Exception as e:
        logger.error(f"Exception during MinerU processing: {e}")
        return []

def run_pipeline():
    """Main pipeline function that can be called directly from a notebook"""
    logger.info("Starting MinerU GCS Pipeline")
    
    # Create local directories
    os.makedirs(Config.LOCAL_TEMP_DIR, exist_ok=True)
    os.makedirs(Config.LOCAL_OUTPUT_DIR, exist_ok=True)
    
    # Ensure we have authentication
    if not ensure_auth():
        logger.error("Failed to authenticate with GCS. Exiting.")
        return
    
    # Get already processed papers
    already_processed = get_already_processed()
    
    # Initialize GCS client
    try:
        storage_client = storage.Client(project=Config.PROJECT_ID)
        logger.info(f"Initialized GCS client for project: {Config.PROJECT_ID}")
        
        source_bucket = storage_client.bucket(Config.SOURCE_BUCKET_NAME)
        dest_bucket = storage_client.bucket(Config.DESTINATION_BUCKET_NAME)
        logger.info(f"Source bucket: {Config.SOURCE_BUCKET_NAME}")
        logger.info(f"Destination bucket: {Config.DESTINATION_BUCKET_NAME}")
    except Exception as e:
        logger.error(f"Failed to initialize GCS client: {e}")
        return
    
    # List all PDF files in the source bucket
    try:
        logger.info(f"Listing PDFs in {Config.SOURCE_BUCKET_NAME}/{Config.SOURCE_PREFIX}")
        blobs = list(source_bucket.list_blobs(prefix=Config.SOURCE_PREFIX))
        pdf_blobs = [blob for blob in blobs if blob.name.lower().endswith('.pdf')]
        
        if not pdf_blobs:
            logger.warning(f"No PDF files found in {Config.SOURCE_BUCKET_NAME}/{Config.SOURCE_PREFIX}")
            return
        
        logger.info(f"Found {len(pdf_blobs)} PDF files")
        
        # Filter out already processed papers
        new_pdf_blobs = []
        for blob in pdf_blobs:
            paper_name = Path(blob.name).stem
            if paper_name not in already_processed:
                new_pdf_blobs.append(blob)
            
        logger.info(f"Found {len(new_pdf_blobs)} new papers to process")
        
        if not new_pdf_blobs:
            logger.info("No new papers to process. Exiting.")
            return
    except Exception as e:
        logger.error(f"Error listing blobs: {e}")
        return
    
    try:
        # Download files in parallel
        logger.info("Downloading PDF files...")
        blob_infos = [(blob.name, storage_client, source_bucket) for blob in new_pdf_blobs]
        
        with ThreadPoolExecutor(max_workers=Config.MAX_WORKERS) as executor:
            local_paths = list(tqdm(
                executor.map(download_blob, blob_infos),
                total=len(new_pdf_blobs),
                desc="Downloading"
            ))
        
        # Process files in batches
        logger.info("Processing PDF files with MinerU vLLM backend...")
        all_processed = []
        for i in range(0, len(local_paths), Config.BATCH_SIZE):
            batch = local_paths[i:i+Config.BATCH_SIZE]
            logger.info(f"Processing batch {i//Config.BATCH_SIZE + 1}/{(len(local_paths) + Config.BATCH_SIZE - 1)//Config.BATCH_SIZE}")
            
            processed_names = process_papers_batch(batch)
            all_processed.extend(processed_names)
            
            # Record processed papers immediately
            record_processed(processed_names)
        
        # Upload processed results
        logger.info("Uploading processed files...")
        total_uploaded = 0
        
        upload_infos = []
        for paper_name in all_processed:
            paper_output_dir = Config.LOCAL_OUTPUT_DIR / paper_name
            if paper_output_dir.exists():
                upload_infos.append((paper_output_dir, paper_name, storage_client, dest_bucket))
        
        with ThreadPoolExecutor(max_workers=Config.MAX_WORKERS) as executor:
            results = list(tqdm(
                executor.map(upload_folder, upload_infos),
                total=len(upload_infos),
                desc="Uploading"
            ))
            
            total_uploaded = sum(results)
        
        logger.info(f"Pipeline complete: Processed {len(all_processed)} papers, uploaded {total_uploaded} files")
    except Exception as e:
        logger.error(f"Error in pipeline: {e}")
    finally:
        # Cleanup
        if Config.DO_CLEANUP:
            logger.info("Cleaning up temporary files...")
            try:
                shutil.rmtree(Config.LOCAL_TEMP_DIR)
                shutil.rmtree(Config.LOCAL_OUTPUT_DIR)
                logger.info("Cleanup complete")
            except Exception as e:
                logger.error(f"Error during cleanup: {e}")
        else:
            logger.info("Skipping cleanup as requested")

# When running as a script (not in a notebook)
if __name__ == "__main__":
    # If you still want to support command line use, use a try block to catch parsing errors
    try:
        import argparse
        parser = argparse.ArgumentParser(description='Process papers from GCS with MinerU')
        parser.add_argument('--batch-size', type=int, help='Number of papers to process in each batch')
        parser.add_argument('--workers', type=int, help='Number of MinerU worker threads')
        parser.add_argument('--no-cleanup', action='store_true', help='Skip cleanup of temporary files')
        
        args, unknown = parser.parse_known_args()
        
        # Override config with command line arguments
        if args.batch_size:
            Config.BATCH_SIZE = args.batch_size
        if args.workers:
            Config.MINERU_WORKERS = args.workers
        if args.no_cleanup:
            Config.DO_CLEANUP = False
    except:
        pass  # Ignore parsing errors in notebooks
    
    run_pipeline()

2025-11-02 12:16:28,119 - mineru-pipeline - INFO - Starting MinerU GCS Pipeline
2025-11-02 12:16:28,120 - mineru-pipeline - ERROR - GCS authentication failed: File path/to/your-service-account-key.json was not found.
2025-11-02 12:16:28,121 - mineru-pipeline - INFO - Attempting to authenticate with gcloud...
2025-11-02 12:16:28,133 - mineru-pipeline - ERROR - Failed to authenticate using gcloud
2025-11-02 12:16:28,134 - mineru-pipeline - ERROR - Failed to authenticate with GCS. Exiting.


In [ ]:
def test_mineru_single_file():
    """Test MinerU with a single file to ensure it's working"""
    # Create test directories
    test_input = Path("./papers")
    test_output = Path("./test_output")
    os.makedirs(test_input, exist_ok=True)
    os.makedirs(test_output, exist_ok=True)
    
    # Find an existing PDF to test with
    test_files = list(Config.LOCAL_TEMP_DIR.glob("*.pdf"))
    if test_files:
        test_file = test_files[0]
        logger.info(f"Testing MinerU with file: {test_file}")
        
        # Try with pipeline backend
        cmd = f"mineru -p {test_file} -o {test_output} -b pipeline"
        logger.info(f"Running: {cmd}")
        
        try:
            result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
            logger.info(f"Return code: {result.returncode}")
            logger.info(f"Output: {result.stdout[:500]}")
            logger.info(f"Errors: {result.stderr[:500]}")
            
            # Check for output files
            output_files = list(test_output.glob("**/*"))
            logger.info(f"Generated {len(output_files)} output files")
            
            # List the files
            for f in output_files:
                logger.info(f"- {f}")
                
            return len(output_files) > 0
        except Exception as e:
            logger.error(f"Error during MinerU test: {e}")
            return False
    else:
        logger.error("No PDF files found to test with")
        return False
    
test_mineru_single_file()

NameError: name 'Path' is not defined